In [ ]:
import pandas as pd

df = pd.read_csv("data/StudentPerformanceFactors.csv")

# Remover linhas com qualquer valor vazio para manter a integridade
df = df.dropna().copy()

df.head()


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [ ]:
# Risco de Presença: <70 (2 pts), <85 (1 pt), else 0
def categorizar_presenca(valor):
    if valor < 70: return 2
    if valor < 85: return 1
    return 0
df['Attendance_Risk'] = df['Attendance'].apply(categorizar_presenca)

def calcular_tendencia_avancada(linha):
    atual = linha['Exam_Score']
    anterior = linha['Previous_Scores']
    
    # Calculando a variação percentual
    # Evitamos divisão por zero caso a nota anterior seja 0
    variacao = (atual - anterior) / anterior if anterior > 0 else 0
    
    # REGRA 1: Nota abaixo de 60 é risco crítico independente da evolução
    if atual < 60:
        return 1
    
    # REGRA 2: Aumento expressivo (> 15%) ganha bônus de proteção
    if variacao >= 0.15:
        return -1
    
    # REGRA 3: Queda expressiva (< -15%) gera ponto de risco
    if variacao <= -0.15:
        return 1
    
    # REGRA 4: Estabilidade ou oscilação pequena (entre -15% e +15%)
    # Se a nota for > 60 (já checado acima), o risco é neutro
    return 0

# Aplicando a nova lógica
df['Trend_Risk'] = df.apply(calcular_tendencia_avancada, axis=1)

print("✅ Lógica de Momentum (Variação %) aplicada com sucesso!")

# Tutoring: 0 sessões (1 pt de risco), >0 sessões (0 pts)
df['Tutoring_Risk'] = df['Tutoring_Sessions'].apply(lambda x: 1 if x == 0 else 0)

# Mapeamentos Inversos (Low=2, Medium=1, High=0)
mapa_inv = {'Low': 2, 'Medium': 1, 'High': 0}
df['Parental_Risk'] = df['Parental_Involvement'].map(mapa_inv)
df['Resources_Risk'] = df['Access_to_Resources'].map(mapa_inv)
df['Motivation_Risk'] = df['Motivation_Level'].map(mapa_inv)
df['Teacher_Risk'] = df['Teacher_Quality'].map(mapa_inv)

# Outros Fatores
df['Disability_Risk'] = df['Learning_Disabilities'].map({'Yes': 1, 'No': 0})
df['Distance_Risk'] = df['Distance_from_Home'].map({'Near': 0, 'Moderate': 1, 'Far': 2})

# --- FATOR DE PROTEÇÃO (SUBTRAI PONTOS) ---
# Atividades Extras: Sim (-1 pt), Não (0 pt)
df['Extracurricular_Bonus'] = df['Extracurricular_Activities'].map({'Yes': -1, 'No': 0})

# --- CÁLCULO FINAL ---

# Soma máxima teórica de riscos = 15 pontos
# (Parental 2 + Resources 2 + Motivation 2 + Attendance 2 + Trend 1 + Teacher 2 + Disability 1 + Distance 2 + Tutoring 1)
max_pontos = 12

df['Vulnerability_Points'] = (
    df['Parental_Risk'] + df['Resources_Risk'] + df['Motivation_Risk'] + 
    df['Attendance_Risk'] + df['Trend_Risk'] + df['Teacher_Risk'] + 
    df['Disability_Risk'] + df['Distance_Risk'] + df['Tutoring_Risk'] + 
    df['Extracurricular_Bonus']
)

df['Vulnerability_Points'] = df['Vulnerability_Points'].clip(lower=0, upper=max_pontos)

# Probabilidade 0-100%
df['Probability_of_Issue'] = (df['Vulnerability_Points'] / max_pontos) * 100
df['Probability_of_Issue'] = df['Probability_of_Issue'].round(2)

# Categorização de Risco
def definir_categoria(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    if prob <= 80: return 'Alto'
    return 'Crítico'

df['Risk_Level'] = df['Probability_of_Issue'].apply(definir_categoria)
colunas_visao = ['Teacher_Quality', 'Tutoring_Sessions', 'Extracurricular_Activities', 'Previous_Scores', 'Exam_Score', 'Parental_Involvement', 'Motivation_Level', 'Attendance', 'Probability_of_Issue']

# Visualização do Top 10 Alunos em Risco
df[colunas_visao].sort_values(by='Probability_of_Issue', ascending=False).head(10)

✅ Lógica de Momentum (Variação %) aplicada com sucesso!


,Teacher_Quality,Tutoring_Sessions,Extracurricular_Activities,Previous_Scores,Exam_Score,Parental_Involvement,Motivation_Level,Attendance,Probability_of_Issue
368,Medium,0,No,100,63,Medium,Low,68,80.00
3543,Medium,0,Yes,66,57,Low,Low,67,80.00
2893,Low,0,Yes,71,59,Medium,Low,60,80.00
5168,Medium,0,Yes,84,59,Low,Low,64,80.00
4774,Medium,0,No,93,58,Medium,Low,73,73.33
283,Medium,0,Yes,68,59,Medium,Low,69,73.33
419,Medium,0,No,74,62,Medium,Low,65,73.33
3789,Low,0,Yes,89,64,Low,Medium,64,73.33
3493,Medium,0,No,66,58,Medium,Medium,60,73.33
3263,Low,0,No,54,61,Low,Medium,67,73.33


In [62]:
# Lista de colunas para conferência
colunas_momentum = [
    'Previous_Scores', 'Exam_Score', 'Trend_Risk',
    'Attendance', 'Attendance_Risk',
    'Parental_Involvement', 'Parental_Risk',
    'Extracurricular_Activities', 'Extracurricular_Bonus',
    'Probability_of_Issue', 'Risk_Level'
]

# Recalculamos os pontos totais porque o Trend_Risk mudou
df['Vulnerability_Points'] = (
    df['Parental_Risk'] + df['Resources_Risk'] + df['Motivation_Risk'] + 
    df['Attendance_Risk'] + df['Trend_Risk'] + df['Teacher_Risk'] + 
    df['Disability_Risk'] + df['Distance_Risk'] + df['Tutoring_Risk'] + 
    df['Extracurricular_Bonus']
)
df['Vulnerability_Points'] = df['Vulnerability_Points'].clip(lower=0, upper=15)
df['Probability_of_Issue'] = (df['Vulnerability_Points'] / 15 * 100).round(2)

# Exibindo os 3 primeiros transpostos para sua interpretação
df.sort_values(by='Probability_of_Issue', ascending=False)[colunas_momentum].head(3)

,Previous_Scores,Exam_Score,Trend_Risk,Attendance,Attendance_Risk,Parental_Involvement,Parental_Risk,Extracurricular_Activities,Extracurricular_Bonus,Probability_of_Issue,Risk_Level
368,100,63,1,68,2,Medium,1,No,0,80.0,Alto
3543,66,57,1,67,2,Low,2,Yes,-1,80.0,Alto
2893,71,59,1,60,2,Medium,1,Yes,-1,80.0,Alto


In [3]:
# A soma máxima possível de todos os riscos mapeados é 15
max_pontos = 15

# Somamos todos os riscos e aplicamos o bônus negativo
df['Vulnerability_Points'] = (
    df['Parental_Risk'] + df['Resources_Risk'] + df['Motivation_Risk'] + 
    df['Attendance_Risk'] + df['Trend_Risk'] + df['Teacher_Risk'] + 
    df['Disability_Risk'] + df['Distance_Risk'] + df['Tutoring_Risk'] + 
    df['Extracurricular_Bonus']
)

# Travamos o valor entre 0 e 15 (para não ter risco negativo ou maior que 100%)
df['Vulnerability_Points'] = df['Vulnerability_Points'].clip(lower=0, upper=max_pontos)

# Transformamos a pontuação bruta em uma probabilidade de 0 a 100%
df['Probability_of_Issue'] = (df['Vulnerability_Points'] / max_pontos) * 100
df['Probability_of_Issue'] = df['Probability_of_Issue'].round(2)

In [ ]:

def definir_categoria(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    if prob <= 80: return 'Alto'
    return 'Crítico'

df['Risk_Level'] = df['Probability_of_Issue'].apply(definir_categoria)

# Selecionamos as colunas principais para a sua análise final
colunas_visao = [
    'Teacher_Quality', 'Tutoring_Sessions', 'Extracurricular_Activities', 
    'Previous_Scores', 'Exam_Score', 'Parental_Involvement', 
    'Motivation_Level', 'Attendance', 'Probability_of_Issue', 'Risk_Level'
]

# 1. Organizamos a lista completa de comparação (Original vs Risco)
# Cada linha aqui representa um dos fatores que somam os 15 pontos
colunas_full_detalhe = [
    # Fatores Qualitativos (0 a 2 pts)
    'Parental_Involvement', 'Parental_Risk',
    'Access_to_Resources', 'Resources_Risk',
    'Motivation_Level', 'Motivation_Risk',
    'Teacher_Quality', 'Teacher_Risk',
    'Distance_from_Home', 'Distance_Risk',
    
    # Fatores de Frequência e Histórico (0 a 2 pts / 0 a 1 pt)
    'Attendance', 'Attendance_Risk',
    'Previous_Scores', 'Exam_Score', 'Trend_Risk',
    
    # Fatores Binários e Saúde (0 a 1 pt)
    'Tutoring_Sessions', 'Tutoring_Risk',
    'Learning_Disabilities', 'Disability_Risk',
    
    # Fator de Proteção (Bônus -1 pt)
    'Extracurricular_Activities', 'Extracurricular_Bonus',
    
    # Resultados Finais
    'Probability_of_Issue', 'Risk_Level'
]

# 2. Ordenamos pelo maior risco
df_ordenado = df.sort_values(by='Probability_of_Issue', ascending=False)

# 3. Exibimos os 3 primeiros com todas as colunas de rastro
df_full_3 = df_ordenado[colunas_full_detalhe].head(3)

print("--- AUDITORIA COMPLETA DE DADOS: VALOR BRUTO VS PESO DE RISCO ---")

display(df_full_3.T)

df_ordenado = df.sort_values(by='Probability_of_Issue', ascending=True)
df_full_3 = df_ordenado[colunas_full_detalhe].head(3)
display(df_full_3.T)

--- AUDITORIA COMPLETA DE DADOS: VALOR BRUTO VS PESO DE RISCO ---


,368,3543,2893
Parental_Involvement,Medium,Low,Medium
Parental_Risk,1,2,1
Access_to_Resources,Low,Low,Medium
Resources_Risk,2,2,1
Motivation_Level,Low,Low,Low
Motivation_Risk,2,2,2
Teacher_Quality,Medium,Medium,Low
Teacher_Risk,1,1,2
Distance_from_Home,Far,Far,Far
Distance_Risk,2,2,2


,6522,1549,1620
Parental_Involvement,High,High,High
Parental_Risk,0,0,0
Access_to_Resources,High,Medium,Medium
Resources_Risk,0,1,1
Motivation_Level,Low,Medium,High
Motivation_Risk,2,1,0
Teacher_Quality,High,High,High
Teacher_Risk,0,0,0
Distance_from_Home,Near,Near,Near
Distance_Risk,0,0,0


In [8]:
def get_student_diagnostic(aluno_row):
    causas = []
    
    # Verificando cada fator para "explicar" o risco
    if aluno_row['Attendance_Risk'] == 2:
        causas.append("baixa frequência escolar (crítica)")
    if aluno_row['Motivation_Risk'] == 2:
        causas.append("baixa motivação intrínseca")
    if aluno_row['Teacher_Risk'] == 2:
        causas.append("percepção de baixa qualidade do corpo docente")
    if aluno_row['Tutoring_Risk'] == 1:
        causas.append("falta de acompanhamento em tutorias/reforço")
    if aluno_row['Trend_Risk'] == 1:
        causas.append("queda recente no desempenho acadêmico")
    if aluno_row['Parental_Risk'] == 2:
        causas.append("baixo envolvimento dos responsáveis")

    # Construindo a frase
    if not causas:
        return "Aluno apresenta perfil estável com baixo índice de vulnerabilidade."
    
    frase_inicial = f"O aluno possui {aluno_row['Probability_of_Issue']}% de risco. "
    detalhes = "Fatores principais: " + ", ".join(causas) + "."
    
    return frase_inicial + detalhes

# Testando para o primeiro aluno da lista de risco
aluno_exemplo = df.sort_values(by='Probability_of_Issue', ascending=False).iloc[0]
print(get_student_diagnostic(aluno_exemplo))



O aluno possui 80.0% de risco. Fatores principais: baixa frequência escolar (crítica), baixa motivação intrínseca, falta de acompanhamento em tutorias/reforço, queda recente no desempenho acadêmico.


In [ ]:
import pandas as pd
import json

# --- 1. CARREGAMENTO E TRADUÇÃO INICIAL ---
df = pd.read_csv("data/StudentPerformanceFactors.csv").dropna().copy()

df = df.rename(columns={
    'Hours_Studied': 'Horas_Estudo',
    'Parental_Involvement': 'Envolvimento_Pais',
    'Access_to_Resources': 'Acesso_Recursos',
    'Motivation_Level': 'Nivel_Motivacao',
    'Attendance': 'Frequencia',
    'Teacher_Quality': 'Qualidade_Professor',
    'Distance_from_Home': 'Distancia_Casa',
    'Learning_Disabilities': 'Dificuldades_Aprendizado',
    'Tutoring_Sessions': 'Sessoes_Reforco',
    'Extracurricular_Activities': 'Atividades_Extra',
    'Previous_Scores': 'Notas_Anteriores',
    'Exam_Score': 'Nota_Exame'
})

# --- 2. MAPEAMENTOS DE RISCO (PESOS) ---
mapa_inv = {'Low': 2, 'Medium': 1, 'High': 0}

# Sugestão futura limitar a 1 ponto, refletindo o impacto real
mapa_professor = {'Low': 1, 'Medium': 0, 'High': 0}

df['Risco_Familiar'] = df['Envolvimento_Pais'].map(mapa_inv)
df['Risco_Recursos'] = df['Acesso_Recursos'].map(mapa_inv)
df['Risco_Motivacao'] = df['Nivel_Motivacao'].map(mapa_inv)
df['Risco_Professor'] = df['Qualidade_Professor'].map(mapa_professor)
df['Risco_Distancia'] = df['Distancia_Casa'].map({'Near': 0, 'Moderate': 1, 'Far': 2})
df['Risco_Dificuldade'] = df['Dificuldades_Aprendizado'].map({'Yes': 1, 'No': 0})
df['Risco_Reforco'] = df.apply(
    lambda x: 1 if (x['Sessoes_Reforco'] == 0 and x['Nota_Exame'] < 65) else 0,
    axis=1
)
df['Bonus_Extra'] = df['Atividades_Extra'].map({'Yes': -1, 'No': 0})

# --- NOVO: BÔNUS POR HORAS DE ESTUDO ---
def calcular_bonus_estudo(horas):
    if horas >= 30: return -1
    return 0

df['Bonus_Estudo'] = df['Horas_Estudo'].apply(calcular_bonus_estudo)

# Risco de Frequência
df['Risco_Frequencia'] = pd.cut(df['Frequencia'], bins=[0, 70, 85, 100], labels=[2, 1, 0], include_lowest=True).astype(int)

# --- 3. LÓGICA DE MOMENTUM ---
def calcular_risco_tendencia(linha):
    atual = linha['Nota_Exame']
    anterior = linha['Notas_Anteriores']
    variacao = (atual - anterior) / anterior * 100 if anterior > 0 else 0
    
    if atual < 60: return 1       
    if variacao >= 15: return -1  
    return 0                     


df['Risco_Tendencia'] = df.apply(calcular_risco_tendencia, axis=1)

# --- 4. CÁLCULO FINAL DO ÍNDICE ---
# Mantemos max_pontos em 14 como referência de risco base
max_pontos = 14
colunas_soma = [
    'Risco_Familiar', 'Risco_Recursos', 'Risco_Motivacao', 'Risco_Frequencia',
    'Risco_Professor', 'Risco_Distancia', 'Risco_Reforco', 'Risco_Dificuldade',
    'Risco_Tendencia', 'Bonus_Extra', 'Bonus_Estudo' # Adicionado Bonus_Estudo
]

df['Pontos_Vulnerabilidade'] = df[colunas_soma].sum(axis=1).clip(lower=0, upper=max_pontos)
df['Probabilidade_Problema'] = (df['Pontos_Vulnerabilidade'] / max_pontos * 100).round(2)

def categorizar_risco(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    return 'Alto'

df['Nivel_Risco'] = df['Probabilidade_Problema'].apply(categorizar_risco)
# --- 4.1 ÍNDICE ACADÊMICO ---
# Foca em dificuldade de aprendizado: frequência, reforço, tendência da nota
cols_academico = ['Risco_Frequencia', 'Risco_Reforco', 'Risco_Dificuldade', 'Risco_Tendencia', 'Bonus_Estudo']
max_academico  = 6
df['Score_Academico'] = (df[cols_academico].sum(axis=1).clip(0, max_academico) / max_academico * 100).round(2)
df['Risco_Academico'] = df['Score_Academico'].apply(lambda p: 'Baixo' if p <= 20 else ('Médio' if p <= 50 else 'Alto'))

# --- 4.2 ÍNDICE DE EVASÃO ---
# Foca em desengajamento: motivação, apoio familiar, distância, recursos, vínculo
cols_evasao = ['Risco_Motivacao', 'Risco_Familiar', 'Risco_Distancia', 'Risco_Recursos', 'Bonus_Extra']
max_evasao  = 7
df['Score_Evasao'] = (df[cols_evasao].sum(axis=1).clip(0, max_evasao) / max_evasao * 100).round(2)
df['Risco_Evasao'] = df['Score_Evasao'].apply(lambda p: 'Baixo' if p <= 20 else ('Médio' if p <= 50 else 'Alto'))

# --- 5. DIAGNÓSTICO AUTOMATIZADO ---
def gerar_diagnostico(row):
    alertas_acad = []
    alertas_evas = []
    elogios = []

    # --- alertas acadêmicos ---
    if row['Risco_Frequencia'] == 2:  alertas_acad.append("frequência crítica")
    if row['Risco_Tendencia']  == 1:  alertas_acad.append("nota em zona crítica")
    if row['Risco_Reforco']    == 1:  alertas_acad.append("sem reforço e com dificuldade")
    if row['Risco_Professor']  == 1:  alertas_acad.append("necessita suporte pedagógico")

    # --- alertas de evasão ---
    if row['Risco_Motivacao']  == 2:  alertas_evas.append("motivação baixa")
    if row['Risco_Familiar']   == 2:  alertas_evas.append("sem apoio familiar")
    if row['Risco_Distancia']  == 2:  alertas_evas.append("distância crítica")

    # --- elogios (positivos em ambos os contextos) ---
    if row['Risco_Tendencia']  == -1: elogios.append("ótima evolução nas notas")
    if row['Bonus_Extra']      == -1: elogios.append("engajado em atividades extras")
    if row['Bonus_Estudo']     == -1: elogios.append("alta carga de estudo individual")

    msg = f"Risco Geral {row['Probabilidade_Problema']}% | Acadêmico {row['Score_Academico']}% | Evasão {row['Score_Evasao']}%."
    if alertas_acad: msg += f" Acadêmico: {', '.join(alertas_acad)}."
    if alertas_evas: msg += f" Evasão: {', '.join(alertas_evas)}."
    if elogios:      msg += f" Destaque: {', '.join(elogios)}."
    return msg

df['Texto_Diagnostico'] = df.apply(gerar_diagnostico, axis=1)

resumo = {
    "total_alunos": len(df),
    "media_risco_geral": round(df['Probabilidade_Problema'].mean(), 2),
    "contagem_por_risco": df['Nivel_Risco'].value_counts().to_dict(),

    # --- dados para o dashboard de ação ---
    "contagem_por_risco_academico": df['Risco_Academico'].value_counts().to_dict(),
    "contagem_por_risco_evasao":    df['Risco_Evasao'].value_counts().to_dict(),
    "quadrantes": {
        "saudavel":            int(len(df[(df['Risco_Academico']=='Baixo') & (df['Risco_Evasao']=='Baixo')])),
        "dificuldade_academica": int(len(df[(df['Risco_Academico']=='Alto')  & (df['Risco_Evasao']=='Baixo')])),
        "risco_evasao":        int(len(df[(df['Risco_Academico']=='Baixo') & (df['Risco_Evasao']=='Alto')])),
        "intervencao_urgente": int(len(df[(df['Risco_Academico']=='Alto')  & (df['Risco_Evasao']=='Alto')]))
    }
}

pacote_completo = {
    "estatisticas_gerais": resumo,
    "lista_alunos": df[[
        'Envolvimento_Pais', 'Nivel_Motivacao', 'Frequencia', 'Nota_Exame',
        'Probabilidade_Problema', 'Nivel_Risco',
        'Score_Academico', 'Risco_Academico',
        'Score_Evasao', 'Risco_Evasao',
        'Texto_Diagnostico'
    ]].to_dict(orient='records')
}

with open('dashboard_estudantes.json', 'w', encoding='utf-8') as f:
    json.dump(pacote_completo, f, ensure_ascii=False, indent=4)

print("🚀 Script atualizado e JSON gerado!")

# --- 6. AUDITORIA COMPLETA NO IPYNB ---
ordem_risco = ['Baixo', 'Médio', 'Alto']
cols_entradas = ['Horas_Estudo', 'Envolvimento_Pais', 'Frequencia', 'Nota_Exame', 'Atividades_Extra']
cols_pesos = ['Bonus_Estudo', 'Risco_Familiar', 'Risco_Frequencia', 'Risco_Tendencia', 'Bonus_Extra']
cols_final = ['Pontos_Vulnerabilidade', 'Probabilidade_Problema', 'Texto_Diagnostico']

df_auditoria = df.groupby('Nivel_Risco').first()
niveis_reais = [n for n in ordem_risco if n in df_auditoria.index]
df_exemplos = df_auditoria.loc[niveis_reais]

display(df_exemplos[cols_entradas + cols_pesos + cols_final].T.style.background_gradient(
    cmap='YlOrRd', 
    subset=pd.IndexSlice[cols_pesos + ['Pontos_Vulnerabilidade'], :]
))

variacao = (df['Nota_Exame'] - df['Notas_Anteriores']) / df['Notas_Anteriores'] * 100
print(variacao.describe())
print("\nQuantos têm variação <= -15%:", (variacao <= -15).sum())
print("Quantos têm nota < 60:", (df['Nota_Exame'] < 60).sum())
print("---------------------------")

variacao = (df['Nota_Exame'] - df['Notas_Anteriores']) / df['Notas_Anteriores'] * 100
print("Quantos teriam risco com limiar -44%:", (variacao <= -44).sum())
print("Quantos teriam bônus com >= 15%:", (variacao >= 15).sum())

🚀 Script atualizado e JSON gerado!


Nivel_Risco,Baixo,Médio,Alto
Horas_Estudo,24,23,19
Envolvimento_Pais,Medium,Low,Low
Frequencia,98,84,64
Nota_Exame,74,67,61
Atividades_Extra,Yes,No,No
Bonus_Estudo,0,0,0
Risco_Familiar,1,2,2
Risco_Frequencia,0,1,2
Risco_Tendencia,0,0,0
Bonus_Extra,-1,0,0


count    6378.000000
mean       -7.057647
std        18.578927
min       -39.795918
25%       -22.580645
50%       -10.487038
75%         6.349206
max        76.923077
dtype: float64

Quantos têm variação <= -15%: 2649
Quantos têm nota < 60: 66
---------------------------
Quantos teriam risco com limiar -44%: 0
Quantos teriam bônus com >= 15%: 966


In [ ]:
import pandas as pd
import json
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

df = pd.read_csv(os.path.join(BASE_DIR, "data", "StudentPerformanceFactors.csv")).dropna().copy()


df = df.rename(columns={
    'Hours_Studied': 'Horas_Estudo',
    'Parental_Involvement': 'Envolvimento_Pais',
    'Access_to_Resources': 'Acesso_Recursos',
    'Motivation_Level': 'Nivel_Motivacao',
    'Attendance': 'Frequencia',
    'Teacher_Quality': 'Qualidade_Professor',
    'Distance_from_Home': 'Distancia_Casa',
    'Learning_Disabilities': 'Dificuldades_Aprendizado',
    'Tutoring_Sessions': 'Sessoes_Reforco',
    'Extracurricular_Activities': 'Atividades_Extra',
    'Previous_Scores': 'Notas_Anteriores',
    'Exam_Score': 'Nota_Exame'
})

df['ID_Aluno'] = range(1, len(df) + 1)

# --- 2. MAPEAMENTOS DE RISCO (PESOS) ---
mapa_inv = {'Low': 2, 'Medium': 1, 'High': 0}

mapa_professor = {'Low': 1, 'Medium': 0, 'High': 0}

df['Risco_Familiar'] = df['Envolvimento_Pais'].map(mapa_inv)
df['Risco_Recursos'] = df['Acesso_Recursos'].map(mapa_inv)
df['Risco_Motivacao'] = df['Nivel_Motivacao'].map(mapa_inv)
df['Risco_Professor'] = df['Qualidade_Professor'].map(mapa_professor)
df['Risco_Distancia'] = df['Distancia_Casa'].map({'Near': 0, 'Moderate': 1, 'Far': 2})
df['Risco_Dificuldade'] = df['Dificuldades_Aprendizado'].map({'Yes': 1, 'No': 0})
df['Risco_Reforco'] = df.apply(
    lambda x: 1 if (x['Sessoes_Reforco'] == 0 and x['Nota_Exame'] < 65) else 0,
    axis=1
)
df['Bonus_Extra'] = df['Atividades_Extra'].map({'Yes': -1, 'No': 0})

# --- NOVO: BÔNUS POR HORAS DE ESTUDO ---
def calcular_bonus_estudo(horas):
    if horas >= 30: return -1
    return 0

df['Bonus_Estudo'] = df['Horas_Estudo'].apply(calcular_bonus_estudo)

# Risco de Frequência
df['Risco_Frequencia'] = pd.cut(df['Frequencia'], bins=[0, 70, 85, 100], labels=[2, 1, 0], include_lowest=True).astype(int)

# --- 3. LÓGICA DE MOMENTUM ---
def calcular_risco_tendencia(linha):
    atual = linha['Nota_Exame']
    anterior = linha['Notas_Anteriores']
    variacao = (atual - anterior) / anterior * 100 if anterior > 0 else 0
    
    if atual < 60: return 1       
    if variacao >= 15: return -1  
    return 0                      


df['Risco_Tendencia'] = df.apply(calcular_risco_tendencia, axis=1)

# --- 4. CÁLCULO FINAL DO ÍNDICE ---
max_pontos = 14
colunas_soma = [
    'Risco_Familiar', 'Risco_Recursos', 'Risco_Motivacao', 'Risco_Frequencia',
    'Risco_Professor', 'Risco_Distancia', 'Risco_Reforco', 'Risco_Dificuldade',
    'Risco_Tendencia', 'Bonus_Extra', 'Bonus_Estudo' # Adicionado Bonus_Estudo
]

df['Pontos_Vulnerabilidade'] = df[colunas_soma].sum(axis=1).clip(lower=0, upper=max_pontos)
df['Probabilidade_Problema'] = (df['Pontos_Vulnerabilidade'] / max_pontos * 100).round(2)

def categorizar_risco(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    return 'Alto'

df['Nivel_Risco'] = df['Probabilidade_Problema'].apply(categorizar_risco)
# --- 4.1 ÍNDICE ACADÊMICO ---
# Foca em dificuldade de aprendizado: frequência, reforço, tendência da nota
cols_academico = ['Risco_Frequencia', 'Risco_Reforco', 'Risco_Dificuldade', 'Risco_Tendencia', 'Bonus_Estudo']
max_academico  = 6
df['Score_Academico'] = (df[cols_academico].sum(axis=1).clip(0, max_academico) / max_academico * 100).round(2)
df['Risco_Academico'] = df['Score_Academico'].apply(lambda p: 'Baixo' if p <= 20 else ('Médio' if p <= 50 else 'Alto'))

# --- 4.2 ÍNDICE DE EVASÃO ---
# Foca em desengajamento: motivação, apoio familiar, distância, recursos, vínculo
cols_evasao = ['Risco_Motivacao', 'Risco_Familiar', 'Risco_Distancia', 'Risco_Recursos', 'Bonus_Extra']
max_evasao  = 7
df['Score_Evasao'] = (df[cols_evasao].sum(axis=1).clip(0, max_evasao) / max_evasao * 100).round(2)
df['Risco_Evasao'] = df['Score_Evasao'].apply(lambda p: 'Baixo' if p <= 20 else ('Médio' if p <= 50 else 'Alto'))

# --- 5. DIAGNÓSTICO AUTOMATIZADO ---
def gerar_diagnostico(row):
    alertas_acad = []
    alertas_evas = []
    elogios = []

    # --- alertas acadêmicos ---
    if row['Risco_Frequencia'] == 2:  alertas_acad.append("frequência crítica")
    if row['Risco_Tendencia']  == 1:  alertas_acad.append("nota em zona crítica")
    if row['Risco_Reforco']    == 1:  alertas_acad.append("sem reforço e com dificuldade")
    if row['Risco_Professor']  == 1:  alertas_acad.append("necessita suporte pedagógico")

    # --- alertas de evasão ---
    if row['Risco_Motivacao']  == 2:  alertas_evas.append("motivação baixa")
    if row['Risco_Familiar']   == 2:  alertas_evas.append("sem apoio familiar")
    if row['Risco_Distancia']  == 2:  alertas_evas.append("distância crítica")

    # --- elogios (positivos em ambos os contextos) ---
    if row['Risco_Tendencia']  == -1: elogios.append("ótima evolução nas notas")
    if row['Bonus_Extra']      == -1: elogios.append("engajado em atividades extras")
    if row['Bonus_Estudo']     == -1: elogios.append("alta carga de estudo individual")

    msg = f"Risco Geral {row['Probabilidade_Problema']}% | Acadêmico {row['Score_Academico']}% | Evasão {row['Score_Evasao']}%."
    if alertas_acad: msg += f" Acadêmico: {', '.join(alertas_acad)}."
    if alertas_evas: msg += f" Evasão: {', '.join(alertas_evas)}."
    if elogios:      msg += f" Destaque: {', '.join(elogios)}."
    return msg

df['Texto_Diagnostico'] = df.apply(gerar_diagnostico, axis=1)


# --- FATORES DE RISCO E PROTEÇÃO ---
nomes = {
    'Risco_Frequencia':  'Frequência',
    'Risco_Motivacao':   'Motivação',
    'Risco_Familiar':    'Apoio Familiar',
    'Risco_Recursos':    'Acesso Recursos',
    'Risco_Distancia':   'Distância',
    'Risco_Dificuldade': 'Dificuldade',
    'Risco_Reforco':     'Sem Reforço',
    'Risco_Professor':   'Professor',
    'Risco_Tendencia':   'Tendência Nota',
    'Bonus_Extra':       'Ativ. Extras',
    'Bonus_Estudo':      'Horas Estudo',
}

# Gráfico 1 soma bruta separada em risco e proteção
fatores_risco = []
fatores_protecao = []

for col, label in nomes.items():
    total = int(df[col].sum())
    if total >= 0:
        fatores_risco.append({"fator": label, "total": total})
    else:
        fatores_protecao.append({"fator": label, "total": abs(total)})

# Gráfico 2 contagem por nível de cada fator
niveis_originais = {
    'Risco_Frequencia':  {0: 'Alta', 1: 'Média', 2: 'Baixa'},
    'Risco_Motivacao':   {0: 'Alta', 1: 'Média', 2: 'Baixa'},
    'Risco_Familiar':    {0: 'Alto', 1: 'Médio', 2: 'Baixo'},
    'Risco_Recursos':    {0: 'Alto', 1: 'Médio', 2: 'Baixo'},
    'Risco_Distancia':   {0: 'Perto', 1: 'Moderada', 2: 'Longe'},
    'Risco_Dificuldade': {0: 'Não', 1: 'Sim'},
    'Risco_Reforco':     {0: 'Com reforço', 1: 'Sem reforço'},
    'Risco_Professor':   {0: 'Med/Alto', 1: 'Baixo'},
    'Risco_Tendencia':   {-1: 'Melhora', 0: 'Estável', 1: 'Crítico'},
    'Bonus_Extra':       {-1: 'Participa', 0: 'Não participa'},
    'Bonus_Estudo':      {-1: '≥30h', 0: '<30h'},
}

fatores_contagem_niveis = []
for col, label in nomes.items():
    contagem = df[col].value_counts().sort_index().to_dict()
    mapa = niveis_originais[col]
    entry = {"fator": label}
    for val, qtd in contagem.items():
        entry[mapa.get(val, str(val))] = int(qtd)
    fatores_contagem_niveis.append(entry)

# Traduções
mapa_traducao = {
    'Low': 'Baixo', 'Medium': 'Médio', 'High': 'Alto',
    'Near': 'Perto', 'Moderate': 'Moderada', 'Far': 'Longe',
    'Yes': 'Sim', 'No': 'Não',
}

for col in ['Envolvimento_Pais', 'Nivel_Motivacao', 'Qualidade_Professor',
            'Distancia_Casa','Acesso_Recursos', 'Dificuldades_Aprendizado', 'Atividades_Extra']:
    df[col] = df[col].map(mapa_traducao).fillna(df[col])

resumo = {
    "total_alunos": len(df),
    "media_risco_geral": round(df['Probabilidade_Problema'].mean(), 2),
    "contagem_por_risco": df['Nivel_Risco'].value_counts().to_dict(),
    "contagem_por_risco_academico": df['Risco_Academico'].value_counts().to_dict(),
    "contagem_por_risco_evasao":    df['Risco_Evasao'].value_counts().to_dict(),
    "quadrantes": {
        "saudavel":              int(len(df[(df['Risco_Academico']=='Baixo') & (df['Risco_Evasao']=='Baixo')])),
        "dificuldade_academica": int(len(df[(df['Risco_Academico']=='Alto')  & (df['Risco_Evasao']=='Baixo')])),
        "risco_evasao":          int(len(df[(df['Risco_Academico']=='Baixo') & (df['Risco_Evasao']=='Alto')])),
        "intervencao_urgente":   int(len(df[(df['Risco_Academico']=='Alto')  & (df['Risco_Evasao']=='Alto')])),
    },
    "fatores_risco":             fatores_risco,       # ← novo
    "fatores_protecao":          fatores_protecao,    # ← novo
    "fatores_contagem_niveis":   fatores_contagem_niveis,  # ← novo
}

# Adiciona Score_Academico, Risco_Academico, Score_Evasao, Risco_Evasao
pacote_completo = {
    "estatisticas_gerais": resumo,
    "lista_alunos": df[[
        'ID_Aluno',
        'Risco_Familiar',
        'Risco_Recursos',
        'Risco_Distancia',
        'Risco_Frequencia',
        'Risco_Tendencia',
        'Risco_Motivacao',
        'Bonus_Extra',
        'Envolvimento_Pais', 
        'Distancia_Casa', 'Acesso_Recursos',
        'Atividades_Extra','Nivel_Motivacao', 'Frequencia', 'Nota_Exame',
        'Probabilidade_Problema', 'Nivel_Risco',
        'Score_Academico', 'Risco_Academico',
        'Score_Evasao', 'Risco_Evasao',
        'Texto_Diagnostico'
    ]].to_dict(orient='records')
}

output_path = os.path.join(BASE_DIR, 'dashboard_estudantes.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(pacote_completo, f, ensure_ascii=False, indent=4)
    
print("Script atualizado e JSON gerado!")



📊 Auditoria de 360º: Validando todos os fatores de vulnerabilidade


Nivel_Risco,Baixo,Médio,Alto
Horas_Estudo,24,23,19
Envolvimento_Pais,Medium,Low,Low
Acesso_Recursos,Medium,High,Medium
Nivel_Motivacao,Medium,Low,Low
Frequencia,98,84,64
Qualidade_Professor,Medium,Medium,Medium
Distancia_Casa,Near,Near,Moderate
Dificuldades_Aprendizado,No,No,No
Sessoes_Reforco,2,0,2
Atividades_Extra,Yes,No,No
